In [1]:
# Import the library to mount Google Drive
from google.colab import drive

import h5py
import numpy as np
import torch
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import pearsonr, spearmanr
from scipy.signal import correlate
from scipy.stats import wasserstein_distance
from collections import defaultdict
from tqdm import tqdm
from pathlib import Path

In [2]:
# Mount the Google Drive at /content/drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
file_name_RRUFF = "RRUFF_low_bkg_full_intensity_cleaned.hdf5"
file_path_RRUFF = f"/content/drive/MyDrive/Research/OrgA/OrgA Team Shared Folder/Data - HDF5/{file_name_RRUFF}"
data_path_RRUFF = "/content/drive/MyDrive/Research/OrgA/OrgA Team Shared Folder/Data - HDF5/"

In [4]:
file_name_SYNTH = "10Million_pyxtal_RRUFFF-type_repacked_sampled_01.hdf5"
file_path_SYNTH = f"/content/drive/MyDrive/Research/OrgA/OrgA Team Shared Folder/Data - HDF5/{file_name_SYNTH}"
data_path_SYNTH = "/content/drive/MyDrive/Research/OrgA/OrgA Team Shared Folder/Data - HDF5/"

In [5]:

# =====================================================
# CONFIG
# =====================================================
RRUFF_H5 = file_path_RRUFF
SYNTH_H5 = file_path_SYNTH

OUT_DIR  = Path("/content/drive/MyDrive/Research/OrgA/OrgA Team Shared Folder/Data - HDF5/xrd_comparison_outputs")
PLOT_DIR = OUT_DIR / "per_class_plots"
CSV_PATH = OUT_DIR / "per_class_metrics.csv"

USE_CLS  = False
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)


# =========================
# OPTIONAL: LOAD ViT MODEL
# =========================
if USE_CLS:
    from vit_model import ViTXRD   # your ViT definition
    model = ViTXRD(...)
    model.load_state_dict(torch.load("model.pt", map_location=DEVICE))
    model.to(DEVICE)
    model.eval()

    @torch.no_grad()
    def extract_cls(x_np):
        x = torch.tensor(x_np, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        emb = model(x, return_embedding=True)
        emb = F.normalize(emb, dim=1)
        return emb.squeeze(0).cpu().numpy()
else:
    extract_cls = None

# =========================
# HELPERS
# =========================
def normalize(x):
    x = x - np.min(x)
    s = np.sum(x)
    return x / s if s > 0 else x

def safe_cosine(x, y):
    try:
        return float(np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y)))
    except:
        return 0.0

def cross_corr(x, y):
    cc = correlate(x, y, mode="full")
    shift = int(cc.argmax() - (len(x) - 1))
    return float(cc.max()), shift

# =========================
# LOAD DATA
# =========================
with h5py.File(RRUFF_H5, "r") as f:
    X_rr = f["X_test"][:]
    y_rr = f["y_test"][:]

with h5py.File(SYNTH_H5, "r") as f:
    X_sy = f["X_train"][:]
    y_sy = f["y_train"][:]

# =========================
# GROUP BY CLASS
# =========================
rr_by_class = defaultdict(list)
sy_by_class = defaultdict(list)

for x, y in zip(X_rr, y_rr):
    rr_by_class[int(y)].append(x)

for x, y in zip(X_sy, y_sy):
    sy_by_class[int(y)].append(x)

classes = sorted(set(rr_by_class) & set(sy_by_class))

# =========================
# MAIN LOOP
# =========================
rows = []

for cls in tqdm(classes, desc="Processing classes"):
    rr = np.array(rr_by_class[cls])
    sy = np.array(sy_by_class[cls])

    rr_mean = normalize(rr.mean(axis=0))
    sy_mean = normalize(sy.mean(axis=0))

    rr_std = rr.std(axis=0)
    sy_std = sy.std(axis=0)

    # safe metrics
    try:
        pearson = pearsonr(rr_mean, sy_mean)[0]
        if np.isnan(pearson): pearson = 0.0
    except:
        pearson = 0.0

    try:
        spearman = spearmanr(rr_mean, sy_mean)[0]
        if np.isnan(spearman): spearman = 0.0
    except:
        spearman = 0.0

    cosine = safe_cosine(rr_mean, sy_mean)

    try:
        wasser = wasserstein_distance(rr_mean, sy_mean)
        if np.isnan(wasser): wasser = 0.0
    except:
        wasser = 0.0

    xcorr_max, xcorr_shift = cross_corr(rr_mean, sy_mean)

    if USE_CLS and extract_cls is not None:
        rr_emb = extract_cls(rr_mean)
        sy_emb = extract_cls(sy_mean)
        cls_cos = safe_cosine(rr_emb, sy_emb)
    else:
        cls_cos = 0.0

    # -----------------------------
    # Realism score
    # -----------------------------
    realism_score = 0.35*cls_cos + 0.25*cosine + 0.2*pearson - 0.2*wasser

    rows.append({
        "class": cls,
        "pearson": pearson,
        "spearman": spearman,
        "cosine": cosine,
        "wasserstein": wasser,
        "xcorr_max": xcorr_max,
        "xcorr_shift": xcorr_shift,
        "cls_cosine": cls_cos,
        "realism_score": realism_score,
        "n_rruff": len(rr),
        "n_synth_train": len(sy),
    })

    # -----------------------------
    # Plot and display inline
    # -----------------------------
    x_axis = np.arange(len(rr_mean))

    plt.figure(figsize=(14,5))
    plt.plot(x_axis, rr_mean, label="RRUFF (mean)", linewidth=2)
    plt.fill_between(x_axis, rr_mean-rr_std, rr_mean+rr_std, alpha=0.3)
    plt.plot(x_axis, sy_mean, label="Synthetic Train (mean)", linewidth=2)
    plt.fill_between(x_axis, sy_mean-sy_std, sy_mean+sy_std, alpha=0.3)
    plt.title(f"Class {cls} | Realism Score = {realism_score:.3f}")
    plt.xlabel("2θ / Q index")
    plt.ylabel("Normalized Intensity")
    plt.legend()
    plt.tight_layout()
    plt.show()  # Colab inline
    plt.savefig(PLOT_DIR / f"class_{cls}.png", dpi=200)
    plt.close()

# =========================
# SAVE CSV
# =========================
df = pd.DataFrame(rows)
df = df.sort_values("realism_score", ascending=False)
df.to_csv(CSV_PATH, index=False)

print("\n=== TOP 10 MOST REALISTIC CLASSES ===")
print(df.head(10)[["class", "realism_score"]])
print("\n=== BOTTOM 10 LEAST REALISTIC CLASSES ===")
print(df.tail(10)[["class", "realism_score"]])
print(f"\nCSV saved to: {CSV_PATH}")
print(f"Plots saved to: {PLOT_DIR}")

Output hidden; open in https://colab.research.google.com to view.